## Remove bad records from the original assay file

Step 1:  Generate the prediction results after using the VRC01 panel data as GT label for deep_hiv model


Step 2: Using the results, find the antibody/env pairs where IC50 values can’t be found in the panel data, and they have high prediction inaccuracy even after correcting the model with panel data.

Step 3: Manually looked through the data and identified records where different studies report quite different results, then excluded them from the assay file 


The listed manually selected records to be removed is saved at "/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/removed_records_comparison.xlsx"

The updated assay file is saved as '/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/assay_removed_bad_records.text'

## Further remove outlier records from the 'assay_removed_bad_records.text'

Step 1: Finds all antibodies that share epitopes with the antibodies in ANTIBODIES_LIST

Step 2: Finds antibody-virus pairs where the same antibody-virus pair (from different publications) has different IC50 measurements that lead to conflicting sensitivity labels conflict across different thresholds

Step 3: Identifies and flags IC50 measurements that contradict the consensus (80% majority) sensitivity classification for each antibody-virus pair. Also tracks how often each Reference (study/source) appears as an outlier.

Step 4: Removes antibody-virus-reference combinations that were identified as high-frequency outliers (those appearing more than 20 times) at 0.2 μg/mL threshold

The updated assay file is saved as 
'/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/assay_removed_bad_records_and_outliers.text'

In [3]:
import pandas as pd
import numpy as np

def virus_is_sensitive(ic50: str, threshold: float) -> bool:
    """Return True if IC50 indicates sensitivity (< threshold)."""
    if ic50.startswith('>'):
        return False
    value = float(ic50[1:]) if ic50.startswith('<') else float(ic50)
    return value < threshold

def parse_ic50(ic50: str) -> float:
    """Convert IC50 string to numeric value."""
    if ic50.startswith(('>', '<')):
        return float(ic50[1:])
    return float(ic50)

def geometric_mean(values):
    """Calculate geometric mean of a series."""
    values = [v for v in values if not pd.isna(v)]
    if not values:
        return np.nan
    return np.exp(np.mean(np.log(values)))

# Antibodies to exclude
# EXCLUDE_ANTIBODIES = [
#     '561_01_18', 'N6', 'PGT121', 'VRC01', 'SF12', '10-1074', 
#     'PGT145', 'M1214_N1', '3BNC117', 'PGDM1400'
# ]
EXCLUDE_ANTIBODIES = []


# Original antibody list to match epitopes
ANTIBODIES_LIST = ['561_01_18', 'N6', 'PGT121', 'VRC01', 'SF12', '10-1074', 'PGT145', 
           'M1214_N1', '3BNC117', 'PGDM1400', 'VRC38.01', 'PG9', '8ANC195', 
           '35O22', 'VRC34.01', 'PGT135', 'b12', 'VRC-PG04', 'PGT151', 'CH01', 
           'DH270.6', '4E10', 'PGT128', 'HJ16', 'VRC26.25', 'VRC-CH31', '10E8', 'VRC07-523-W54-LS.v3']


# Load epitope data
epitope_path = '/home/yujieq/work/ML_training/bNAb-ReP/original_data/env_neu_unique_ab_aligned.txt'
try:
    epitope_df = pd.read_csv(epitope_path, sep='\t')
except:
    try:
        epitope_df = pd.read_csv(epitope_path, sep=',')
    except:
        epitope_df = pd.read_csv(epitope_path, sep='\s+')

# Get epitopes for our antibody list
ab_epitopes = epitope_df[epitope_df['Antibody'].isin(ANTIBODIES_LIST)]['epitope'].unique()

# Expand antibody list (excluding the specified antibodies)
expanded_antibodies = [
    ab for ab in epitope_df[epitope_df['epitope'].isin(ab_epitopes)]['Antibody'].unique() 
    if ab not in EXCLUDE_ANTIBODIES
]

# Load assay data
assay_path = '/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/assay_removed_bad_records.text'
try:
    assay_df = pd.read_csv(assay_path, sep='\t')
except:
    try:
        assay_df = pd.read_csv(assay_path, sep=',')
    except:
        assay_df = pd.read_csv(assay_path, sep='\s+')

# Filter for expanded antibody list (excluding specified antibodies) and non-null IC50
filtered_df = assay_df[
    assay_df['Antibody'].isin(expanded_antibodies) & 
    assay_df['IC50'].notna()
].copy()

# Add numeric IC50 column
filtered_df['IC50_numeric'] = filtered_df['IC50'].apply(parse_ic50)

# Calculate geometric mean for each antibody-virus pair
geo_means = filtered_df.groupby(['Antibody', 'Virus'])['IC50_numeric'].apply(geometric_mean).reset_index()
geo_means.rename(columns={'IC50_numeric': 'Geometric_mean'}, inplace=True)

# Add sensitivity columns
for threshold in [0.2, 1, 50]:
    filtered_df[f'Sensitive_{threshold}'] = filtered_df['IC50'].apply(lambda x: virus_is_sensitive(x, threshold))

# Find antibody-virus pairs with conflicting labels across thresholds
conflicting_pairs = []
for (ab, vs), group in filtered_df.groupby(['Antibody', 'Virus']):
    if (group['Sensitive_0.2'].nunique() > 1 or 
        group['Sensitive_1'].nunique() > 1 or 
        group['Sensitive_50'].nunique() > 1):
        conflicting_pairs.append((ab, vs))

# Filter for conflicting pairs and merge with geometric means
result_df = filtered_df[filtered_df[['Antibody', 'Virus']].apply(tuple, axis=1).isin(conflicting_pairs)].copy()
result_df = pd.merge(result_df, geo_means, on=['Antibody', 'Virus'], how='left')

# Sort and clean up
result_df = result_df.sort_values(['Antibody', 'Virus', 'IC50'])
result_df.drop(columns=['IC50_numeric'], inplace=True)

result_df

/tmp/ipykernel_3571204/2825979616.py:61: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  assay_df = pd.read_csv(assay_path, sep='\t')


,Antibody,Virus,Reference,Pubmed ID,IC50,IC80,ID50,Sensitive_0.2,Sensitive_1,Sensitive_50,Geometric_mean
4,10-1074,0013095_2_11,Chuang et al. Structure 27:196 (2019),30471922.0,4.22,>50,NaN,False,False,True,11.989122
0,10-1074,0013095_2_11,Bonsignori et al. Sci Transl Med 9:eaai7514 (2...,28298420.0,5.84,NaN,NaN,False,False,True,11.989122
2,10-1074,0013095_2_11,Bricault et al. Cell Host Microbe 25:59 (2019)...,30629920.0,5.84,NaN,NaN,False,False,True,11.989122
6,10-1074,0013095_2_11,Huang et al. Immunity 45:1108 (2016a) - dataset 1,27851912.0,5.84,NaN,NaN,False,False,True,11.989122
8,10-1074,0013095_2_11,Kwon et al. Cell Reports 22:1798 (2018),29444432.0,5.84,>50,NaN,False,False,True,11.989122
...,...,...,...,...,...,...,...,...,...,...,...
10091,m66.6,MN_3,Ofek et al. J Virol 88:2426 (2014),24335316.0,0.303,NaN,NaN,False,True,True,1.424907
10090,m66.6,MN_3,Georgiev et al. Science 340:751 (2013) - datas...,23661761.0,3.09,NaN,NaN,False,False,True,1.424907
10092,m66.6,MN_3,Zhu et al. J Virol 85:1140 (2011),21880764.0,3.09,NaN,NaN,False,False,True,1.424907
10094,m66.6,T257_31,Zhu et al. J Virol 85:1140 (2011),21880764.0,10.6,NaN,NaN,False,False,True,23.021729


In [4]:
# Save the results
output_path = '/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/conflicting_sensitivity_records.xlsx'

with pd.ExcelWriter(output_path) as writer:
    result_df.to_excel(writer, sheet_name='all_same_epitope_antibodies', index=False)

In [5]:
import pandas as pd
import numpy as np
from collections import Counter

def get_majority_label(series, threshold=0.8):
    """Return majority label if it appears in >= threshold proportion, else None"""
    counts = series.value_counts()
    total = len(series)
    for label, count in counts.items():
        if count / total >= threshold:
            return label
    return None  # No clear majority

def find_outliers(group, threshold_col):
    """Find rows where threshold_col contradicts the 80% majority in group"""
    majority = get_majority_label(group[threshold_col])
    if majority is None:  # No clear majority - return empty DataFrame
        return pd.DataFrame(columns=group.columns)
    outliers = group[group[threshold_col] != majority].copy()
    return outliers

# Load the data
input_path = '/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/conflicting_sensitivity_records.xlsx'
df = pd.read_excel(input_path)

# Process for threshold 0.2
outliers_02 = df.groupby(['Antibody', 'Virus']).apply(find_outliers, 'Sensitive_0.2').reset_index(drop=True)
# Count frequency of each Reference across ALL groups
if not outliers_02.empty:
    ref_counts_02 = Counter(outliers_02['Reference'])
    outliers_02['outlier_freq'] = outliers_02['Reference'].map(ref_counts_02)
    outliers_02 = outliers_02.sort_values(['outlier_freq', 'Antibody', 'Virus'], ascending=[False, True, True])

# Process for threshold 1
outliers_1 = df.groupby(['Antibody', 'Virus']).apply(find_outliers, 'Sensitive_1').reset_index(drop=True)
# Count frequency of each Reference across ALL groups
if not outliers_1.empty:
    ref_counts_1 = Counter(outliers_1['Reference'])
    outliers_1['outlier_freq'] = outliers_1['Reference'].map(ref_counts_1)
    outliers_1 = outliers_1.sort_values(['outlier_freq', 'Antibody', 'Virus'], ascending=[False, True, True])

# Select columns to keep
output_cols = ['Antibody', 'Virus', 'Reference', 'IC50', 'Geometric_mean', 
               'Sensitive_0.2', 'Sensitive_1', 'outlier_freq']

# Save back to original file with new tabs
with pd.ExcelWriter(input_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    if not outliers_02.empty:
        outliers_02[output_cols].to_excel(writer, sheet_name='outlier_thres0.2', index=False)
    else:
        pd.DataFrame(columns=output_cols).to_excel(writer, sheet_name='outlier_thres0.2', index=False)
    
    if not outliers_1.empty:
        outliers_1[output_cols].to_excel(writer, sheet_name='outlier_thres1', index=False)
    else:
        pd.DataFrame(columns=output_cols).to_excel(writer, sheet_name='outlier_thres1', index=False)

print("Outlier analysis completed with strict 80% majority rule:")
print(f"- Groups with clear majority at 0.2 threshold: {outliers_02['Antibody'].nunique() if not outliers_02.empty else 0}")
print(f"- Groups with clear majority at 1 threshold: {outliers_1['Antibody'].nunique() if not outliers_1.empty else 0}")

Outlier analysis completed with strict 80% majority rule:
- Groups with clear majority at 0.2 threshold: 35
- Groups with clear majority at 1 threshold: 37


In [6]:
import pandas as pd

# Load the original assay data
assay_path = '/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/assay_removed_bad_records.text'
try:
    assay_df = pd.read_csv(assay_path, sep='\t')
except:
    try:
        assay_df = pd.read_csv(assay_path, sep=',')
    except:
        assay_df = pd.read_csv(assay_path, sep='\s+')

# Load the outlier records
outlier_path = '/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/conflicting_sensitivity_records.xlsx'
outliers_02 = pd.read_excel(outlier_path, sheet_name='outlier_thres0.2')

# Get the high-frequency outliers (freq > 20)
high_freq_outliers = outliers_02[outliers_02['outlier_freq'] > 20]
high_freq_triplets = high_freq_outliers[['Antibody', 'Virus', 'Reference']].drop_duplicates()

# Create a mask to identify rows to keep (not in high-frequency outliers)
mask = assay_df.merge(high_freq_triplets, 
                     on=['Antibody', 'Virus', 'Reference'],
                     how='left', 
                     indicator=True)['_merge'] == 'left_only'

# Filter the original assay data
cleaned_df = assay_df[mask].copy()

# Save the cleaned data
output_path = '/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/assay_removed_bad_records_and_outliers.text'
cleaned_df.to_csv(output_path, sep='\t', index=False)

# Print summary
original_count = len(assay_df)
cleaned_count = len(cleaned_df)
removed_count = original_count - cleaned_count
outlier_triplets_count = len(high_freq_triplets)

print(f"Original records: {original_count}")
print(f"Removed {removed_count} records matching {outlier_triplets_count} high-frequency outlier triplets")
print(f"Cleaned data saved to: {output_path}")

/tmp/ipykernel_3571204/1500753716.py:6: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  assay_df = pd.read_csv(assay_path, sep='\t')


Original records: 171624
Removed 277 records matching 277 high-frequency outlier triplets
Cleaned data saved to: /home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/assay_removed_bad_records_and_outliers.text


# Identify and filter out the specified duplicate strain pairs from the assay file

Identifies all records that contain any of the duplicate strains

Returns a DataFrame containing only these records

In [5]:
import pandas as pd
from pathlib import Path

# Define file paths
input_file = Path("/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/assay_removed_bad_records_and_outliers.text")

# List of all duplicate strains (both sides of each pair)
duplicate_strains = {
    "093_TF_SGA17", "093_AT_SGA1",
    "1536_V15_D11_4", "1536_BL_E11_4",
    "1536_BL_E6_1", "1536_BL_E5_3",
    "1HD2_PREINF_051916_D8_S66", "1HD2_PREINF_051916_A12_S61",
    "2305_V7_G6_1", "2305_V7_G2_2",
    "2305_V7_H5_1", "2305_V7_H11_2",
    "2936_V9_B3_2", "2936_V9_A3_3",
    "6292_V7_G8_4", "6292_V7_G10_4",
    "651_AT_SGA5", "651_AT_SGA2",
    "651_LN_C43", "651_LN_C40",
    "693_1969_V11_C4_1_REBOUND", "693_1969_V11_A7_1_REBOUND",
    "693_7989_V9_B2_2_REBOUND", "693_7989_V9_A4_1_REBOUND",
    "CH040_C", "700010040_C9_4520",
    "CH058_C", "700010058_A4_4375",
    "703010054_2A2_REV", "703010054_2A2",
    "703010217_B6_REV", "703010217_B6",
    "704809221_1B3_REV", "704809221_1B3",
    "706010164_1A7_REV", "706010164_1A7",
    "8074_BL_F2_4", "8074_BL_D10_2",
    "8074_V7_E4_4", "8074_V7_E2_2",
    "9252_PRE_H7", "9252_PRE_E14",
    "9252_REBOUND_S71_D3", "9252_REBOUND_S70_D5",
    "9341_W5_S38_B5", "9341_W5_S36_B7",
    "AIIMS704_70402", "AIIMS704_70401",
    "AIIMS706_70607B", "AIIMS706_70604B",
    "AIIMS706_70606Z", "AIIMS706_70606A",
    "AIIMS706_70607F", "AIIMS706_70607A",
    "CE2010_F5_REV", "CE2010_F5",
    "CNE56", "CNE107",
    "CNE28", "CNE26",
    "EN3C068", "EN3C016",
    "EN3C084", "EN3C040",
    "EN3D079", "EN3D020",
    "QF495_23M_A3", "QF495_23M_A1",
    "SHIV_CNE40_JB98_W29_3", "SHIV_CNE40_JB98_W29_2",
    "V704_0847_030_ESN_02C", "V704_0847_030_ESN_01T",
    "V704_2524_240ESN_G", "V704_2524_240ESN_A",
    "V704_2777_210_SN_02A", "V704_2777_210_SN_01G",
    "V704_2901_190ESN_02C", "V704_2901_190ESN_01T",
    "ZM246F_REV", "ZM246F"
}

# Read the input file
df = pd.read_csv(input_file, sep='\t')

# Find all records containing duplicate strains
duplicate_records = df[df['Virus'].isin(duplicate_strains)]

# Print summary
print(f"Found {len(duplicate_records)} records containing duplicate strains")

# Return the DataFrame of duplicate records (without modifying original file)
duplicate_records

Found 3274 records containing duplicate strains


/tmp/ipykernel_3048935/3041563487.py:52: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_file, sep='\t')


,Antibody,Virus,Reference,Pubmed ID,IC50,IC80,ID50
7,0.5gamma,CH040_C,Ramirez Valdez et al. Virology 475:187 (2015),25486586.0,>150,NaN,NaN
8,0.5gamma,CH058_C,Ramirez Valdez et al. Virology 475:187 (2015),25486586.0,>150,NaN,NaN
106,0PV-a.01,703010217_B6,Kong et al. Cell 178:567-584.e19 (2019),31348886.0,>500,>500,NaN
165,0PV-a.01,CNE56,Kong et al. Cell 178:567-584.e19 (2019),31348886.0,24.6,201,NaN
315,0PV-b.01,703010217_B6,Kong et al. Cell 178:567-584.e19 (2019),31348886.0,>500,>500,NaN
...,...,...,...,...,...,...,...
171248,PGT128+VRC07,703010217_B6,Kong et al. J Virol 89:2659 (2015) - dataset 1,25520506.0,0.011,0.042,NaN
171249,PGT128+VRC07,704809221_1B3,Kong et al. J Virol 89:2659 (2015) - dataset 1,25520506.0,0.021,0.062,NaN
171274,PGT128+VRC07,CE2010_F5,Kong et al. J Virol 89:2659 (2015) - dataset 1,25520506.0,0.156,0.483,NaN
171329,PGT128+VRC07,ZM246F,Kong et al. J Virol 89:2659 (2015) - dataset 1,25520506.0,0.007,0.02,NaN


#### Now for those filtered "duplicate_records", group them under the same identical pair of strains call them A and B (e.g. "093_TF_SGA17" and "093_AT_SGA1", "1536_V15_D11_4" and "1536_BL_E11_4") for each antibody. Note that there might be multiple records for the same pair of "antibody "and "virus" due to column "Reference" might be different. Then under each group, examine if the IC50 for identical strain A and  for identical strain B falls on the opposite site of cutoff (0.2, 1, and 50). For example, if IC50 for "093_TF_SGA17" is 0.06 and if IC50 for "093_AT_SGA1" is 0.3 / if IC50 for "093_TF_SGA17" is 0.06 and if IC50 for "093_AT_SGA1" is 1.5  / if IC50 for "093_TF_SGA17" is 0.8 and if IC50 for "093_AT_SGA1" is 20 / if IC50 for "093_TF_SGA17" is 30 and if IC50 for "093_AT_SGA1" is >50, those scenarios they all meet the condition  " identical strain A and  for identical strain B falls on the opposite site of cutoff (0.2, 1, and 50)".    Create a new df called "duplicates_to_remove" and append all records that meet the condition onto this new df. 

This script:

Creates a dictionary mapping each strain to its identical pair

Parses IC50 values into numeric format, handling > and < prefixes

For each strain pair:

Finds records for both strains

Identifies antibodies that appear in both strains

Compares IC50 values across all combinations

Checks if values fall on opposite sides of any cutoff (0.2, 1, or 50)

Adds qualifying records to the removal dataframe

Returns the final dataframe with all records that should be removed

The output will show all records where identical strain pairs have IC50 values that disagree across the specified cutoffs.

In [8]:
import numpy as np

# Create a dictionary mapping each strain to its pair
strain_pairs = [
    ("093_TF_SGA17", "093_AT_SGA1"),
    ("1536_V15_D11_4", "1536_BL_E11_4"),
    ("1536_BL_E6_1", "1536_BL_E5_3"),
    ("1HD2_PREINF_051916_D8_S66", "1HD2_PREINF_051916_A12_S61"),
    ("2305_V7_G6_1", "2305_V7_G2_2"),
    ("2305_V7_H5_1", "2305_V7_H11_2"),
    ("2936_V9_B3_2", "2936_V9_A3_3"),
    ("6292_V7_G8_4", "6292_V7_G10_4"),
    ("651_AT_SGA5", "651_AT_SGA2"),
    ("651_LN_C43", "651_LN_C40"),
    ("693_1969_V11_C4_1_REBOUND", "693_1969_V11_A7_1_REBOUND"),
    ("693_7989_V9_B2_2_REBOUND", "693_7989_V9_A4_1_REBOUND"),
    ("CH040_C", "700010040_C9_4520"),
    ("CH058_C", "700010058_A4_4375"),
    ("703010054_2A2_REV", "703010054_2A2"),
    ("703010217_B6_REV", "703010217_B6"),
    ("704809221_1B3_REV", "704809221_1B3"),
    ("706010164_1A7_REV", "706010164_1A7"),
    ("8074_BL_F2_4", "8074_BL_D10_2"),
    ("8074_V7_E4_4", "8074_V7_E2_2"),
    ("9252_PRE_H7", "9252_PRE_E14"),
    ("9252_REBOUND_S71_D3", "9252_REBOUND_S70_D5"),
    ("9341_W5_S38_B5", "9341_W5_S36_B7"),
    ("AIIMS704_70402", "AIIMS704_70401"),
    ("AIIMS706_70607B", "AIIMS706_70604B"),
    ("AIIMS706_70606Z", "AIIMS706_70606A"),
    ("AIIMS706_70607F", "AIIMS706_70607A"),
    ("CE2010_F5_REV", "CE2010_F5"),
    ("CNE56", "CNE107"),
    ("CNE28", "CNE26"),
    ("EN3C068", "EN3C016"),
    ("EN3C084", "EN3C040"),
    ("EN3D079", "EN3D020"),
    ("QF495_23M_A3", "QF495_23M_A1"),
    ("SHIV_CNE40_JB98_W29_3", "SHIV_CNE40_JB98_W29_2"),
    ("V704_0847_030_ESN_02C", "V704_0847_030_ESN_01T"),
    ("V704_2524_240ESN_G", "V704_2524_240ESN_A"),
    ("V704_2777_210_SN_02A", "V704_2777_210_SN_01G"),
    ("V704_2901_190ESN_02C", "V704_2901_190ESN_01T"),
    ("ZM246F_REV", "ZM246F")
]

# Convert IC50 to numeric values, handling > and < prefixes
def parse_ic50(ic50_str):
    try:
        if pd.isna(ic50_str):
            return np.nan
        ic50_str = str(ic50_str).strip()
        if ic50_str.startswith('>'):
            return float(ic50_str[1:])
        elif ic50_str.startswith('<'):
            return float(ic50_str[1:])
        else:
            return float(ic50_str)
    except ValueError:
        return np.nan

duplicate_records['IC50_num'] = duplicate_records['IC50'].apply(parse_ic50)

# Initialize dataframe to store records to remove
duplicates_to_remove = pd.DataFrame()

# Check each strain pair for opposite-side cutoff cases
for strain_a, strain_b in strain_pairs:
    # Get records for both strains
    records_a = duplicate_records[duplicate_records['Virus'] == strain_a]
    records_b = duplicate_records[duplicate_records['Virus'] == strain_b]
    
    # Skip if either strain has no records
    if records_a.empty or records_b.empty:
        continue
    
    # For each antibody that appears in both strains
    common_antibodies = set(records_a['Antibody']).intersection(set(records_b['Antibody']))
    for antibody in common_antibodies:
        # Get IC50 values for this antibody in both strains
        ic50_a = records_a[records_a['Antibody'] == antibody]['IC50_num'].values
        ic50_b = records_b[records_b['Antibody'] == antibody]['IC50_num'].values
        
        # Compare all combinations of IC50 values between the strains
        for a in ic50_a:
            for b in ic50_b:
                if not (np.isnan(a) or np.isnan(b)):
                    # Check if they fall on opposite sides of any cutoff
                    for cutoff in [0.2, 1, 50]:
                        if (a < cutoff and b >= cutoff) or (a >= cutoff and b < cutoff):
                            # Add both records to removal list
                            dup_a = records_a[(records_a['Antibody'] == antibody) & 
                                            (records_a['IC50_num'] == a)]
                            dup_b = records_b[(records_b['Antibody'] == antibody) & 
                                            (records_b['IC50_num'] == b)]
                            duplicates_to_remove = pd.concat([duplicates_to_remove, dup_a, dup_b])
                            break  # No need to check other cutoffs if one matches

# Clean up - remove temporary column and drop duplicates
duplicates_to_remove = duplicates_to_remove.drop(columns=['IC50_num']).drop_duplicates()

print(f"Found {len(duplicates_to_remove)} records with conflicting IC50 values across strain pairs")
duplicates_to_remove

Found 183 records with conflicting IC50 values across strain pairs


,Antibody,Virus,Reference,Pubmed ID,IC50,IC80,ID50
5186,10E8,093_TF_SGA17,Mandizvo et al. J Virol 96:e0127022 (2022),36453881.0,0.640,NaN,NaN
5178,10E8,093_AT_SGA1,Mandizvo et al. J Virol 96:e0127022 (2022),36453881.0,1.017,NaN,NaN
92713,PGDM1400,1536_V15_D11_4,Stephenson et al. Nat Med 7:1718 (2021),34621054.0,0.201,1.237,NaN
92706,PGDM1400,1536_BL_E11_4,Stephenson et al. Nat Med 7:1718 (2021),34621054.0,0.057,0.54,NaN
24809,3BNC117,1536_BL_E6_1,Stephenson et al. Nat Med 7:1718 (2021),34621054.0,0.192,0.891,NaN
...,...,...,...,...,...,...,...
20177,2F5,ZM246F_REV,Montefiori CAVD CA-VIMC,NaN,>25,NaN,NaN
20172,2F5,ZM246F,Bricault et al. Cell Host Microbe 25:59 (2019)...,30629920.0,>50,NaN,NaN
20173,2F5,ZM246F,Bricault et al. Cell Host Microbe 25:59 (2019)...,30629920.0,>50,NaN,NaN
20174,2F5,ZM246F,Hraber et al. AIDS 28:163 (2014),24361678.0,>50,NaN,NaN


#### Now we've found the initial set of duplicates_to_remove from "duplicate_records". For those remaining ones in "duplicate_records" (those with similar IC50), remove the strain with less antibody records across all assay file records. For example, if A and B are identical, A has data for 10 antibodies in assay file, B has data for 5 antibodies, add all records associated with B in "duplicate_records" to the df "duplicates_to_remove". If an antibody only has A or B, that should be ok to keep all the record so don't add them to "duplicates_to_remove". if A and B has the same number of antibodies in assay file, add all records associated with both A and B in "duplicate_records" to the df "duplicates_to_remove"

This script:

Identifies remaining duplicate records not already marked for removal

Counts how many unique antibodies each strain has in the full assay file

For each strain pair:

Compares the antibody counts

Removes the strain with fewer antibodies (or both if counts are equal)

Adds these records to the duplicates_to_remove dataframe

Preserves records where an antibody only has data for one strain in a pair

This ensures we:

Keep the strain with more comprehensive antibody coverage

Only remove complete strain records when they're truly redundant

Maintain all unique antibody-strain combinations

The output will show all records that should be removed, now including both:

Records with conflicting IC50 values across cutoffs (from previous step)

Records from strains with less antibody coverage (from this step)

In [9]:
# First get the remaining duplicate records not already in duplicates_to_remove
remaining_duplicates = duplicate_records[~duplicate_records.index.isin(duplicates_to_remove.index)]

# Count number of antibodies per strain in the full assay file
strain_ab_counts = df.groupby('Virus')['Antibody'].nunique()

# Initialize list to store additional records to remove
additional_to_remove = []

# Process each strain pair
for strain_a, strain_b in strain_pairs:
    # Skip if either strain not in remaining duplicates
    if strain_a not in remaining_duplicates['Virus'].values or strain_b not in remaining_duplicates['Virus'].values:
        continue
    
    # Get antibody counts from full assay file
    count_a = strain_ab_counts.get(strain_a, 0)
    count_b = strain_ab_counts.get(strain_b, 0)
    
    # Determine which strain(s) to remove
    if count_a > count_b:
        # Remove strain B
        to_remove = remaining_duplicates[remaining_duplicates['Virus'] == strain_b]
    elif count_b > count_a:
        # Remove strain A
        to_remove = remaining_duplicates[remaining_duplicates['Virus'] == strain_a]
    else:
        # Counts equal, remove both
        to_remove = remaining_duplicates[remaining_duplicates['Virus'].isin([strain_a, strain_b])]
    
    additional_to_remove.append(to_remove)

# Combine all additional records to remove
if additional_to_remove:
    additional_to_remove = pd.concat(additional_to_remove)
    duplicates_to_remove = pd.concat([duplicates_to_remove, additional_to_remove])
    print(f"Added {len(additional_to_remove)} additional records to remove based on antibody counts")
else:
    print("No additional records to remove based on antibody counts")

# Final count
print(f"Total records to remove: {len(duplicates_to_remove)}")
duplicates_to_remove

Added 609 additional records to remove based on antibody counts
Total records to remove: 792


,Antibody,Virus,Reference,Pubmed ID,IC50,IC80,ID50,IC50_num
5186,10E8,093_TF_SGA17,Mandizvo et al. J Virol 96:e0127022 (2022),36453881.0,0.640,NaN,NaN,NaN
5178,10E8,093_AT_SGA1,Mandizvo et al. J Virol 96:e0127022 (2022),36453881.0,1.017,NaN,NaN,NaN
92713,PGDM1400,1536_V15_D11_4,Stephenson et al. Nat Med 7:1718 (2021),34621054.0,0.201,1.237,NaN,NaN
92706,PGDM1400,1536_BL_E11_4,Stephenson et al. Nat Med 7:1718 (2021),34621054.0,0.057,0.54,NaN,NaN
24809,3BNC117,1536_BL_E6_1,Stephenson et al. Nat Med 7:1718 (2021),34621054.0,0.192,0.891,NaN,NaN
...,...,...,...,...,...,...,...,...
155130,VRC26.25-LS,V704_2901_190ESN_02C,Montefiori HVTN unpublished data,NaN,>25,>25,NaN,25.00
22761,2G12,ZM246F_REV,Montefiori CAVD CA-VIMC,NaN,21.33,NaN,NaN,21.33
34965,4E10,ZM246F_REV,Montefiori CAVD CA-VIMC,NaN,2.98,NaN,NaN,2.98
44885,b12,ZM246F_REV,Montefiori CAVD CA-VIMC,NaN,>25,NaN,NaN,25.00


In [10]:
# Define output path for the duplicates to remove
duplicates_remove_file = Path("/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/duplicates_to_remove.txt")

# Save to tab-delimited text file
duplicates_to_remove.to_csv(duplicates_remove_file, sep='\t', index=False)

print(f"\nSaved {len(duplicates_to_remove)} duplicate records to remove at:")
print(duplicates_remove_file)


Saved 792 duplicate records to remove at:
/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/duplicates_to_remove.txt


# Remove the duplicate records from CATNAP assay file

In [11]:
import pandas as pd
from pathlib import Path

# Define file paths
duplicate_file = Path("/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/duplicates_to_remove.txt")
assay_file = Path("/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/assay_removed_bad_records_and_outliers.text")
output_file = Path("/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/assay_removed_bad_records_and_outliers_duplicates.text")

# Read both files, keeping IC50 as strings
duplicates_df = pd.read_csv(duplicate_file, sep='\t', dtype={'IC50': str})
assay_df = pd.read_csv(assay_file, sep='\t', dtype={'IC50': str})

# Print initial counts
print(f"Initial assay records: {len(assay_df)}")
print(f"Duplicate records to remove: {len(duplicates_df)}")

def format_ic50(ic50_str):
    """Format IC50 string to 3 decimal places if it's a numeric value"""
    try:
        # Try to convert to float, round to 3 decimals, then back to string
        return f"{float(ic50_str):.3f}"
    except (ValueError, TypeError):
        # Return original if not convertible to float
        return ic50_str

# Create a composite key for comparison in both DataFrames
for df in [duplicates_df, assay_df]:
    # Apply formatting to IC50 while keeping it as string
    formatted_ic50 = df['IC50'].apply(format_ic50)
    df['composite_key'] = (df['Antibody'] + '|' + df['Virus'] + '|' + df['Reference'] + '|' + 
                         df['Pubmed ID'].astype(str) + '|' + formatted_ic50)

# Find records to keep (those not in duplicates)
filtered_assay_df = assay_df[~assay_df['composite_key'].isin(duplicates_df['composite_key'])]

# Drop the temporary composite key column
filtered_assay_df = filtered_assay_df.drop(columns=['composite_key'])

# Ensure IC50 remains as string in final output
filtered_assay_df['IC50'] = filtered_assay_df['IC50'].astype(str)

# Save the filtered data
filtered_assay_df.to_csv(output_file, sep='\t', index=False)

# Print results
print(f"Records after removing duplicates: {len(filtered_assay_df)}")
print(f"Records removed as duplicates: {len(assay_df) - len(filtered_assay_df)}")
print(f"Filtered file saved to: {output_file}")

/tmp/ipykernel_3048935/3928240513.py:11: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  assay_df = pd.read_csv(assay_file, sep='\t', dtype={'IC50': str})


Initial assay records: 171347
Duplicate records to remove: 792
Records after removing duplicates: 170555
Records removed as duplicates: 792
Filtered file saved to: /home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/assay_removed_bad_records_and_outliers_duplicates.text


# Update the neut databse file based on calculated geomean after removing duplicates from assay file

if the unique pairs of "Antibody " & "Virus" in /assay_removed_bad_records_and_outliers_duplicates.text" can't be matched with "Antibody " & "virus" in env_neu_unique_ab_removed_outliers_geomean.txt". then remove those unmatched pairs from env_neu_unique_ab_removed_outliers_geomean.txt. in the end, show me how many pairs of "Antibody " & "virus" in env_neu_unique_ab_removed_outliers_geomean.txt" got updated IC50 value and how many pairs being remove

This gives us a complete picture of:

How many pairs were removed because they didn't exist in the assay file

How many pairs had their IC50 values updated

How many pairs remained unchanged

The output file will only contain antibody-virus pairs that exist in both files, with updated IC50 values where available.

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

def geometric_mean(values):
    """Calculate geometric mean of IC50 values with special handling for > and < prefixes"""
    clean = []
    for v in values:
        if pd.isna(v):
            continue
        v = str(v).strip()
        if v.startswith('>'):
            clean.append(50.0)  # Handle > values
        elif v.startswith('<'):
            clean.append(float(v[1:]))  # Handle < values
        else:
            try:
                clean.append(float(v))  # Regular numeric values
            except ValueError:
                continue  # Skip non-numeric values
    
    if not clean:
        return np.nan
    return round(np.exp(np.mean(np.log(clean))), 2)

# Define file paths
assay_file = Path("/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/assay_removed_bad_records_and_outliers_duplicates.text")
env_file = Path("/home/yujieq/work/ML_training/bNAb-ReP/original_data/env_neu_unique_ab_removed_outliers_geomean.txt")
output_file = Path("/home/yujieq/work/ML_training/bNAb-ReP/original_data/env_neu_unique_ab_removed_outliers_duplicates_geomean.txt")

# Read both files
assay_df = pd.read_csv(assay_file, sep='\t', dtype={'IC50': str})
env_df = pd.read_csv(env_file, sep='\t')

# Standardize column names for merging
env_df = env_df.rename(columns={'virus': 'Virus'})  # Make consistent with assay_df

# Calculate geometric mean IC50 for each Antibody+Virus group in assay data
assay_geomean = assay_df.groupby(['Antibody', 'Virus'])['IC50'].apply(geometric_mean).reset_index()
assay_geomean.columns = ['Antibody', 'Virus', 'New_IC50']

# Get unique Antibody+Virus pairs from assay file
assay_pairs = set(zip(assay_geomean['Antibody'], assay_geomean['Virus']))

# Create pair identifiers in env file
env_df['Pair'] = list(zip(env_df['Antibody'], env_df['Virus']))

# Count initial pairs in env file
initial_pairs = len(env_df['Pair'].unique())

# Filter env file to only keep pairs that exist in assay file
filtered_env_df = env_df[env_df['Pair'].isin(assay_pairs)]

# Count removed pairs
removed_pairs = initial_pairs - len(filtered_env_df['Pair'].unique())

# Now proceed with IC50 updates on the filtered data
filtered_env_df['Original_IC50'] = filtered_env_df['IC50']

# Merge with assay geomean data
updated_env_df = filtered_env_df.merge(
    assay_geomean,
    on=['Antibody', 'Virus'],
    how='left'
)

# Update IC50 column where we have new geomean values
updated_env_df['IC50'] = updated_env_df['New_IC50'].combine_first(updated_env_df['IC50'])

# Calculate which records actually changed
updated_env_df['IC50_Changed'] = (updated_env_df['New_IC50'].notna() & 
                                 (updated_env_df['Original_IC50'] != updated_env_df['IC50']))

# Count how many pairs were updated
updated_pairs = len(updated_env_df[updated_env_df['IC50_Changed']]['Pair'].unique())

# Drop temporary columns
updated_env_df = updated_env_df.drop(columns=['New_IC50', 'Original_IC50', 'IC50_Changed', 'Pair'])

# Save the updated file
updated_env_df.to_csv(output_file, sep='\t', index=False)

# Print detailed results
print(f"Updated file saved to: {output_file}")
print(f"\nInitial Antibody+Virus pairs in env file: {initial_pairs}")
print(f"Pairs removed (no match in assay file): {removed_pairs}")
print(f"Pairs remaining after filtering: {initial_pairs - removed_pairs}")
print(f"\nOf the remaining pairs:")
print(f"- Pairs with updated IC50 values: {updated_pairs}")
print(f"- Pairs with unchanged IC50 values: {(initial_pairs - removed_pairs) - updated_pairs}")

/tmp/ipykernel_3048935/3416632783.py:32: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  assay_df = pd.read_csv(assay_file, sep='\t', dtype={'IC50': str})
/tmp/ipykernel_3048935/3416632783.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_env_df['Original_IC50'] = filtered_env_df['IC50']


Updated file saved to: /home/yujieq/work/ML_training/bNAb-ReP/original_data/env_neu_unique_ab_removed_outliers_duplicates_geomean.txt

Initial Antibody+Virus pairs in env file: 65700
Pairs removed (no match in assay file): 534
Pairs remaining after filtering: 65166

Of the remaining pairs:
- Pairs with updated IC50 values: 4
- Pairs with unchanged IC50 values: 65162


# Regenerate CATNAP training file from the processed assay file

given this file "/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/assay_removed_bad_records_and_outliers_duplicates.text"
group by 'Antibody', 'Virus' and Calculate geometric mean of IC50 values with special handling for > and < prefixes like we did before. Then generate a python file which has the below structure:

[[0, "0.5gamma", "6535_3", false], [1, "0.5gamma", "89_6", true], [2, "0.5gamma", "AC10_29", false], [3, "0.5gamma", "BAL_01", true],..., [63827, "vFP7.05", "ZM55_28A", false]]

So in each item, the first element is just the item number ID, the 2nd element is antibody, the 3rd element is virus, the 4th element is label (if geomean IC50 < threshold, then true; otherwise, false;the default is false)

Generate 5 different files based on the below 5 thresholds.

threshold: 0.2, 1, 2, 10, 50

save it as "/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_flat_threshold{threshold}_removed_outliers_duplicates_geomean.json"

In [7]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

def geometric_mean(values):
    """Calculate geometric mean of IC50 values with special handling for > and < prefixes"""
    clean = []
    for v in values:
        if pd.isna(v):
            continue
        v = str(v).strip()
        if v.startswith('>'):
            clean.append(50.0)  # Handle > values
        elif v.startswith('<'):
            clean.append(float(v[1:]))  # Handle < values
        else:
            try:
                clean.append(float(v))  # Regular numeric values
            except ValueError:
                continue  # Skip non-numeric values
    
    if not clean:
        return np.nan
    return round(np.exp(np.mean(np.log(clean))), 2)

# Define input and output paths
input_file = Path("/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/assay_removed_bad_records_and_outliers_duplicates.text")
output_dir = Path("/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/")

# Thresholds to process
thresholds = [0.2, 1, 2, 10, 50]

# Read the input file
df = pd.read_csv(input_file, sep='\t', dtype={'IC50': str})

# Calculate geometric mean for each Antibody+Virus group
geomean_df = df.groupby(['Antibody', 'Virus'])['IC50'].apply(geometric_mean).reset_index()
geomean_df.columns = ['Antibody', 'Virus', 'GeoMean_IC50']

# Track NA records that will be removed
na_records = geomean_df[geomean_df['GeoMean_IC50'].isna()]
print(f"\nFound {len(na_records)} records with NA values after geometric mean calculation:")
print("These records will be removed from the output files.\n")

# Show NA records by antibody if there are any
if not na_records.empty:
    print("NA records by antibody (VRC01 and PGDM1400 only):")
    # Filter for specific antibodies first
    specific_abs = na_records[na_records['Antibody'].isin(["VRC01", "PGDM1400"])]
    if not specific_abs.empty:
        print(specific_abs.groupby('Antibody').size().sort_values(ascending=False))
    else:
        print("No NA records found for VRC01 or PGDM1400")
    print("\n")

# Remove NA records
geomean_df = geomean_df.dropna(subset=['GeoMean_IC50'])

# Process each threshold
for threshold in thresholds:
    # Create the output data structure
    output_data = []
    
    # Assign labels based on threshold
    for idx, row in enumerate(geomean_df.itertuples()):
        label = row.GeoMean_IC50 < threshold
        output_data.append([idx, row.Antibody, row.Virus, label])
    
    # Create output filename
    output_file = output_dir / f"catnap_flat_threshold{threshold}_removed_outliers_duplicates_geomean.json"
    
    # Save as JSON
    with open(output_file, 'w') as f:
        json.dump(output_data, f)
    
    print(f"Saved {len(output_data)} records to {output_file} (after removing NA records)")

print("\nAll threshold files generated successfully.")

/tmp/ipykernel_3172116/3226516249.py:35: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_file, sep='\t', dtype={'IC50': str})



Found 4993 records with NA values after geometric mean calculation:
These records will be removed from the output files.

NA records by antibody (VRC01 and PGDM1400 only):
Antibody
PGDM1400    133
VRC01       133
dtype: int64


Saved 97570 records to /home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_flat_threshold0.2_removed_outliers_duplicates_geomean.json (after removing NA records)
Saved 97570 records to /home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_flat_threshold1_removed_outliers_duplicates_geomean.json (after removing NA records)
Saved 97570 records to /home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_flat_threshold2_removed_outliers_duplicates_geomean.json (after removing NA records)
Saved 97570 records to /home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_flat_threshold10_removed_outliers_duplicates_geomean.json (after removing NA records)
Saved 97570 records to /

## compare "/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_flat_threshold0.2_removed_outliers_duplicates_geomean.json" with "/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_flat_threshold0.2_removed_outliers_geomean.json" 

- first remove all records in "catnap_flat_threshold0.2_removed_outliers_duplicates_geomean.json" where there isn't a match of antibody and virus pairs in catnap_flat_threshold0.2_removed_outliers_geomean.json" 
- then for the remaining records in "catnap_flat_threshold0.2_removed_outliers_duplicates_geomean.json", show how many true/false labels got changed by matching the antibody and virus (in each item of both json files, the first element is just the item number ID, the 2nd element is antibody, the 3rd element is virus, the 4th element is label (if geomean IC50 < threshold, then true; otherwise, false;the default is false)) with the pairs "catnap_flat_threshold0.2_removed_outliers_geomean.json" 
- Lastly, show how many pairs of antibody and virus disappear from the "catnap_flat_threshold0.2_removed_outliers_geomean.json"  to "catnap_flat_threshold0.2_removed_outliers_duplicates_geomean.json"

This script will:

First filter the duplicates file to only include antibody-virus pairs that exist in the outliers file, showing how many records were removed (we do this because we are only including the antibodies and virus sequence appeared in "work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/heavy_seqs_aa.fasta" "work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/light_seqs_aa.fasta"
"work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_data/virus_seqs_aa.fasta"
* #### ****** Note: please refer to dataset.py to see details of this inclusion criteria ***

Compare the labels (true/false) for matching pairs between the two files, showing:

How many changed from true to false

How many changed from false to true

How many remained unchanged

Show how many antibody-virus pairs disappeared when going from the outliers file to the duplicates file

* #### ****** Note: this script handles thresholds 2 and 10 by using the pairs from threshold 1 as reference when their corresponding outliers files don't exist

In [12]:
import json
from pathlib import Path

def compare_and_filter_files(threshold, reference_threshold=None):
    # Define file paths
    base_dir = Path("/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/")
    duplicates_file = base_dir / f"catnap_flat_threshold{threshold}_removed_outliers_duplicates_geomean.json"
    
    # Determine which outliers file to use as reference
    if reference_threshold is None:
        outliers_file = base_dir / f"catnap_flat_threshold{threshold}_removed_outliers_geomean.json"
    else:
        outliers_file = base_dir / f"catnap_flat_threshold{reference_threshold}_removed_outliers_geomean.json"
    
    filtered_output_file = base_dir / f"catnap_flat_threshold{threshold}_removed_outliers_duplicates_geomean.json"

    # Load duplicates file
    with open(duplicates_file, 'r') as f:
        duplicates_data = json.load(f)

    # Load outliers file if it exists, otherwise use reference threshold
    try:
        with open(outliers_file, 'r') as f:
            outliers_data = json.load(f)
        used_reference = False
    except FileNotFoundError:
        if reference_threshold is None:
            # Try with threshold 1 as fallback
            fallback_file = base_dir / f"catnap_flat_threshold1_removed_outliers_geomean.json"
            try:
                with open(fallback_file, 'r') as f:
                    outliers_data = json.load(f)
                used_reference = True
                print(f"Using threshold 1 as reference for threshold {threshold}")
            except FileNotFoundError:
                raise FileNotFoundError(f"Neither {outliers_file} nor {fallback_file} found")
        else:
            raise FileNotFoundError(f"Reference file {outliers_file} not found")

    # Create dictionaries for quick lookup
    duplicates_dict = {(item[1], item[2]): item[3] for item in duplicates_data}
    outliers_dict = {(item[1], item[2]): item[3] for item in outliers_data}

    # 1. Filter duplicates data to only include pairs in outliers file
    filtered_duplicates = [item for item in duplicates_data if (item[1], item[2]) in outliers_dict]

    print(f"\nProcessing threshold {threshold}")
    if used_reference:
        print(f"Using threshold {reference_threshold if reference_threshold else 1} as reference")
    print("="*50)
    print(f"Original duplicates file had {len(duplicates_data)} records")
    print(f"After filtering to only pairs in reference file: {len(filtered_duplicates)} records")
    print(f"Number of pairs removed: {len(duplicates_data) - len(filtered_duplicates)}\n")

    # 2. Compare label changes for matching pairs
    label_changes = {'true_to_false': 0, 'false_to_true': 0, 'unchanged': 0}

    for (ab, virus), dup_label in duplicates_dict.items():
        if (ab, virus) in outliers_dict:
            outlier_label = outliers_dict[(ab, virus)]
            if dup_label == outlier_label:
                label_changes['unchanged'] += 1
            elif dup_label and not outlier_label:
                label_changes['true_to_false'] += 1
            elif not dup_label and outlier_label:
                label_changes['false_to_true'] += 1

    print("Label changes for matching antibody-virus pairs:")
    print(f"True -> False: {label_changes['true_to_false']}")
    print(f"False -> True: {label_changes['false_to_true']}")
    print(f"Unchanged: {label_changes['unchanged']}\n")

    # 3. Show pairs that disappeared from outliers to duplicates
    outliers_pairs = set(outliers_dict.keys())
    duplicates_pairs = set(duplicates_dict.keys())
    disappeared_pairs = outliers_pairs - duplicates_pairs

    print(f"Number of pairs in reference file: {len(outliers_pairs)}")
    print(f"Number of pairs in duplicates file: {len(duplicates_pairs)}")
    print(f"Number of pairs that disappeared: {len(disappeared_pairs)}")

    # Save the filtered duplicates data
    with open(filtered_output_file, 'w') as f:
        json.dump(filtered_duplicates, f)
    print(f"\nSaved filtered data to {filtered_output_file}")

# Process all thresholds with specific references for missing files
thresholds = [
    (0.2, None),  # Use its own outliers file
    (1, None),    # Use its own outliers file
    (2, 1),       # Use threshold 1 as reference
    (10, 1),      # Use threshold 1 as reference
    (50, None)    # Use its own outliers file
]

for threshold, reference in thresholds:
    compare_and_filter_files(threshold, reference)

print("\nAll thresholds processed successfully!")


Processing threshold 0.2
Original duplicates file had 97570 records
After filtering to only pairs in reference file: 63294 records
Number of pairs removed: 34276

Label changes for matching antibody-virus pairs:
True -> False: 0
False -> True: 0
Unchanged: 63294

Number of pairs in reference file: 63828
Number of pairs in duplicates file: 97570
Number of pairs that disappeared: 534

Saved filtered data to /home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_flat_threshold0.2_removed_outliers_duplicates_geomean.json

Processing threshold 1
Original duplicates file had 97570 records
After filtering to only pairs in reference file: 63294 records
Number of pairs removed: 34276

Label changes for matching antibody-virus pairs:
True -> False: 1
False -> True: 0
Unchanged: 63293

Number of pairs in reference file: 63828
Number of pairs in duplicates file: 97570
Number of pairs that disappeared: 534

Saved filtered data to /home/yujieq/work/ML_training/deep_hiv_ab_pre

from ""/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_flat_threshold0.2_removed_outliers_duplicates_geomean.json" ", count the number of true vs. false labels for the below antibodies and return a table:

antibodies = ['561_01_18', 'N6', 'PGT121', 'VRC01', 'SF12', '10-1074', 'PGT145', 
           'M1214_N1', '3BNC117', 'PGDM1400', 'VRC38.01', 'PG9', '8ANC195', 
           '35O22', 'VRC34.01', 'PGT135', 'b12', 'VRC-PG04', 'PGT151', 'CH01', 
           'DH270.6', '4E10', 'PGT128', 'HJ16', 'VRC26.25', 'VRC-CH31', '10E8', 'VRC07-523-W54-LS.v3']

## Create catnap data with epitope&lineage info

In [13]:
import json
import pandas as pd
import math
from pathlib import Path

# Base paths
base_dir = Path("/home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/")
txt_file_path = Path("/home/yujieq/work/ML_training/bNAb-ReP/original_data/env_neu_unique_ab_removed_outliers_geomean.txt")

# Thresholds to process
thresholds = [0.2, 1, 2, 10, 50]

# Load the txt file data once
env_data = pd.read_csv(txt_file_path, delimiter='\t')

# Create mappings
epitope_mapping = {(row['Antibody'], row['virus']): row['epitope'] for _, row in env_data.iterrows()}
lineage_mapping = {(row['Antibody'], row['virus']): row['lineage'] for _, row in env_data.iterrows()}

def process_threshold(threshold):
    # Input and output paths
    input_json = base_dir / f"catnap_flat_threshold{threshold}_removed_outliers_duplicates_geomean.json"
    output_json = base_dir / f"catnap_flat_threshold{threshold}_removed_outliers_duplicates_geomean_with_epitope&lineage.json"
    
    # Load JSON data
    with open(input_json, 'r') as f:
        json_data = json.load(f)
    
    # Add epitope and lineage to each entry
    for entry in json_data:
        antibody, virus = entry[1], entry[2]
        epitope = epitope_mapping.get((antibody, virus))
        lineage = lineage_mapping.get((antibody, virus))
        
        # Handle None/NaN values
        epitope = None if pd.isna(epitope) else epitope
        lineage = None if pd.isna(lineage) else lineage
        
        entry.append(epitope)
        entry.append(lineage)
    
    # Save updated JSON
    with open(output_json, 'w') as f:
        json.dump(json_data, f, indent=4)
    
    print(f"Processed threshold {threshold} - saved to {output_json}")

# Process all thresholds
for threshold in thresholds:
    process_threshold(threshold)

print("\nAll thresholds processed successfully!")

Processed threshold 0.2 - saved to /home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_flat_threshold0.2_removed_outliers_duplicates_geomean_with_epitope&lineage.json
Processed threshold 1 - saved to /home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_flat_threshold1_removed_outliers_duplicates_geomean_with_epitope&lineage.json
Processed threshold 2 - saved to /home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_flat_threshold2_removed_outliers_duplicates_geomean_with_epitope&lineage.json
Processed threshold 10 - saved to /home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_flat_threshold10_removed_outliers_duplicates_geomean_with_epitope&lineage.json
Processed threshold 50 - saved to /home/yujieq/work/ML_training/deep_hiv_ab_pred/deep_hiv_ab_pred/catnap/catnap_flat_threshold50_removed_outliers_duplicates_geomean_with_epitope&lineage.json

All thresholds processed successfully!
